In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, precision_score,
    average_precision_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, precision_recall_curve
)
import mlflow
import mlflow.pytorch
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

ModuleNotFoundError: No module named 'seaborn'

In [15]:
# --- Carregar e preparar dados ---
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "blastchar/telco-customer-churn",
    "WA_Fn-UseC_-Telco-Customer-Churn.csv",
)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce')
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())
df.rename(columns={'Churn': 'target'}, inplace=True)
df['target'] = df['target'].map({'No': 0, 'Yes': 1})

X = df.drop(columns=['customerID', 'target'])
y = df['target']

numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ]
)

print(f"Dataset: {X.shape[0]} amostras, {X.shape[1]} features")
print(f"Target: {y.value_counts(normalize=True).to_dict()}")

Dataset: 7043 amostras, 19 features
Target: {0: 0.7346301292063041, 1: 0.2653698707936959}


In [16]:
class ChurnMLP(nn.Module):
    """
    Multi-Layer Perceptron para predição de churn.
    Arquitetura: Input → 128 → 64 → 32 → 1
    Com BatchNorm e Dropout para regularização.
    """
    def __init__(self, input_dim, dropout_rate=0.3):
        super().__init__()
        self.network = nn.Sequential(
            # Camada 1
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # Camada 2
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # Camada 3
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # Output
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

print("Arquitetura definida: Input → 128 → 64 → 32 → 1")
print("Ativação: ReLU | Regularização: BatchNorm + Dropout(0.3)")
print("Loss: BCELoss (Binary Cross Entropy)")

Arquitetura definida: Input → 128 → 64 → 32 → 1
Ativação: ReLU | Regularização: BatchNorm + Dropout(0.3)
Loss: BCELoss (Binary Cross Entropy)


## Decisões de Arquitetura

- **3 camadas ocultas (128 → 64 → 32)**: rede compacta para dados tabulares (~7k amostras).
  Redes muito profundas não ajudam em datasets tabulares pequenos.
- **ReLU**: ativação padrão — resolve vanishing gradient, computacionalmente eficiente.
- **BatchNorm**: estabiliza o treinamento normalizando as ativações entre camadas.
- **Dropout (0.3)**: regularização para evitar overfitting — desliga 30% dos neurônios a cada batch.
- **Sigmoid na saída**: classificação binária — output entre 0 e 1 (probabilidade de churn).
- **BCELoss**: loss padrão para classificação binária com saída sigmoid.

In [17]:
class EarlyStopping:
    """Para o treinamento quando a val_loss para de melhorar."""
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.should_stop = False
        
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop

In [18]:
def train_mlp(X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor,
              input_dim, epochs=200, batch_size=64, lr=1e-3, dropout_rate=0.3,
              patience=10):
    """
    Treina a MLP com early stopping e retorna modelo + histórico.
    """
    # Criar DataLoaders
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    # Inicializar modelo, loss e otimizador
    model = ChurnMLP(input_dim, dropout_rate=dropout_rate).to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    early_stopping = EarlyStopping(patience=patience)
    history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
    
    for epoch in range(epochs):
        # --- Treinamento ---
        model.train()
        train_losses = []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            y_pred = model(X_batch).squeeze()
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        
        # --- Validação ---
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_tensor.to(device)).squeeze()
            val_loss = criterion(val_pred, y_val_tensor.to(device)).item()
            val_auc = roc_auc_score(
                y_val_tensor.numpy(), 
                val_pred.cpu().numpy()
            )
        
        avg_train_loss = np.mean(train_losses)
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_loss)
        history['val_auc'].append(val_auc)
        
        # Print a cada 20 epochs
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1:3d} | "
                  f"Train Loss: {avg_train_loss:.4f} | "
                  f"Val Loss: {val_loss:.4f} | "
                  f"Val AUC: {val_auc:.4f}")
        
        # Early stopping
        if early_stopping(val_loss):
            print(f"  ⏹ Early stopping na epoch {epoch+1} "
                  f"(melhor val_loss: {early_stopping.best_loss:.4f})")
            break
    
    return model, history

In [19]:
def plot_training_history(history):
    """Plota curvas de loss e AUC durante o treinamento."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(history['train_loss'], label='Train Loss')
    axes[0].plot(history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Curvas de Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    axes[1].plot(history['val_auc'], label='Val AUC-ROC', color='green')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('AUC-ROC')
    axes[1].set_title('AUC-ROC na Validação')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [20]:
# --- Split treino/validação/teste ---
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=SEED, stratify=y_temp
)

print(f"Treino:    {X_train.shape[0]} amostras")
print(f"Validação: {X_val.shape[0]} amostras")
print(f"Teste:     {X_test.shape[0]} amostras")

Treino:    4507 amostras
Validação: 1127 amostras
Teste:     1409 amostras


In [21]:
# --- Pré-processar e converter para tensores ---
preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train).toarray() \
    if hasattr(preprocessor.transform(X_train), 'toarray') \
    else preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val).toarray() \
    if hasattr(preprocessor.transform(X_val), 'toarray') \
    else preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test).toarray() \
    if hasattr(preprocessor.transform(X_test), 'toarray') \
    else preprocessor.transform(X_test)

# Converter para tensores PyTorch
X_train_tensor = torch.FloatTensor(X_train_processed)
y_train_tensor = torch.FloatTensor(y_train.values)
X_val_tensor = torch.FloatTensor(X_val_processed)
y_val_tensor = torch.FloatTensor(y_val.values)
X_test_tensor = torch.FloatTensor(X_test_processed)
y_test_tensor = torch.FloatTensor(y_test.values)

input_dim = X_train_processed.shape[1]
print(f"Input dimension (após encoding): {input_dim} features")

Input dimension (após encoding): 30 features


In [22]:
# --- Treinar MLP ---
print("=" * 50)
print("TREINANDO MLP")
print("=" * 50)

model, history = train_mlp(
    X_train_tensor, y_train_tensor,
    X_val_tensor, y_val_tensor,
    input_dim=input_dim,
    epochs=200,
    batch_size=64,
    lr=1e-3,
    dropout_rate=0.3,
    patience=15
)

plot_training_history(history)

TREINANDO MLP


AttributeError: module 'torch' has no attribute '_utils'

In [ ]:
# --- Avaliar MLP no teste ---
model.eval()
with torch.no_grad():
    y_test_proba_mlp = model(X_test_tensor.to(device)).squeeze().cpu().numpy()
    y_test_pred_mlp = (y_test_proba_mlp >= 0.5).astype(int)

print("=== MLP — Resultados no Teste ===\n")
print(classification_report(y_test, y_test_pred_mlp, target_names=['Sem Churn', 'Com Churn']))

In [ ]:
# --- Treinar modelos de árvore ---
tree_models = {
    'Decision Tree': DecisionTreeClassifier(random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=SEED),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=SEED),
}

# Retreinar baselines também para comparação justa no mesmo split
baseline_models = {
    'DummyClassifier (stratified)': DummyClassifier(strategy='stratified', random_state=SEED),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=SEED),
}

all_sklearn_models = {**baseline_models, **tree_models}
sklearn_results = {}

for name, clf in all_sklearn_models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', clf)])
    pipe.fit(X_train, y_train)
    
    y_proba = pipe.predict_proba(X_test)[:, 1]
    y_pred = pipe.predict(X_test)
    
    sklearn_results[name] = {
        'y_proba': y_proba,
        'y_pred': y_pred,
        'auc_roc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba),
        'f1': f1_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
    }
    print(f"✅ {name} — AUC-ROC: {sklearn_results[name]['auc_roc']:.4f}")